In [1]:
import pandas as pd

train = pd.read_csv("data/twitter_training.csv")

test = pd.read_csv("data/twitter_validation.csv")


In [2]:
train.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [3]:
test.head()

,3364,Facebook,Irrelevant,"I mentioned on Facebook that I was struggling for motivation to go for a run the other day, which has been translated by Tom’s great auntie as ‘Hayley can’t get out of bed’ and told to his grandma, who now thinks I’m a lazy, terrible person 🤣"
0,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
1,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
2,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
3,4433,Google,Neutral,Now the President is slapping Americans in the...
4,6273,FIFA,Negative,Hi @EAHelp I’ve had Madeleine McCann in my cel...


In [4]:
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

train.columns = ["ID", "Entity", "Sentiment", "Text"]
test.columns = ["ID", "Entity", "Sentiment", "Text"]

train.dropna(inplace=True)
test.dropna(inplace=True)

train.drop_duplicates(inplace=True)
test.drop_duplicates(inplace=True)

train = train[train["Sentiment"].isin(["Positive", "Negative"])]
test = test[test["Sentiment"].isin(["Positive", "Negative"])]

train.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [5]:
import re

stop_words = set(stopwords.words("english"))

for word in ["no", "not", "nor"]:
    stop_words.discard(word)

def preprocess(text):

    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"\d+", "", text)

    text = re.sub(r"[^a-z'\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    words = []

    for word in text.split():

        if word not in stop_words:
            words.append(lemmatizer.lemmatize(word))

    return " ".join(words)

In [6]:
train["Clean_Text"] = train["Text"].apply(preprocess)
test["Clean_Text"] = test["Text"].apply(preprocess)

In [7]:
print(train[["Text", "Clean_Text"]].head())

                                                Text  \
0  I am coming to the borders and I will kill you...   
1  im getting on borderlands and i will kill you ...   
2  im coming on borderlands and i will murder you...   
3  im getting on borderlands 2 and i will murder ...   
4  im getting into borderlands and i can murder y...   

                     Clean_Text  
0            coming border kill  
1    im getting borderland kill  
2   im coming borderland murder  
3  im getting borderland murder  
4  im getting borderland murder  


In [8]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

train["Label"] = encoder.fit_transform(train["Sentiment"])
test["Label"] = encoder.transform(test["Sentiment"])

print(encoder.classes_)

['Negative' 'Positive']


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

max_words = 10000

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(train["Clean_Text"])

X_train = tokenizer.texts_to_sequences(train["Clean_Text"])
X_test = tokenizer.texts_to_sequences(test["Clean_Text"])

In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 50

X_train = pad_sequences(
    X_train,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

y_train = train["Label"].values
y_test = test["Label"].values

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [12]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (33128, 50)
Validation: (8282, 50)
Test: (543, 50)


In [13]:
vocab_size = min(max_words, len(tokenizer.word_index) + 1)

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    ),

    SimpleRNN(128),

    Dropout(0.5),

    Dense(64, activation="relu"),

    Dense(len(encoder.classes_), activation="softmax")
])

c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
from tensorflow.keras.layers import Bidirectional

model_bi = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    ),

    Bidirectional(SimpleRNN(128)),

    Dropout(0.5),

    Dense(64, activation="relu"),

    Dense(len(encoder.classes_), activation="softmax")
])

In [16]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [17]:
model_bi.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [18]:
import time
start = time.time()

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val, y_val)
)

simple_time = time.time() - start

Epoch 1/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.7120 - loss: 0.5764 - val_accuracy: 0.7910 - val_loss: 0.4934
Epoch 2/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.8215 - loss: 0.4318 - val_accuracy: 0.8439 - val_loss: 0.3761
Epoch 3/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.8614 - loss: 0.3589 - val_accuracy: 0.7962 - val_loss: 0.5116
Epoch 4/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.8741 - loss: 0.3239 - val_accuracy: 0.7533 - val_loss: 0.5233
Epoch 5/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.7919 - loss: 0.4710 - val_accuracy: 0.8185 - val_loss: 0.4250


In [19]:
start = time.time()

history_bi = model_bi.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val, y_val)
)

bi_time = time.time() - start

Epoch 1/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.7564 - loss: 0.4639 - val_accuracy: 0.8820 - val_loss: 0.2778
Epoch 2/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.9142 - loss: 0.2064 - val_accuracy: 0.8999 - val_loss: 0.2362
Epoch 3/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - accuracy: 0.9480 - loss: 0.1227 - val_accuracy: 0.9079 - val_loss: 0.2273
Epoch 4/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.9555 - loss: 0.0995 - val_accuracy: 0.9055 - val_loss: 0.2436
Epoch 5/5
518/518 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.9626 - loss: 0.0834 - val_accuracy: 0.8995 - val_loss: 0.2761


In [20]:
loss_simple, acc_simple = model.evaluate(X_test, y_test)

print("SimpleRNN")
print("Training Time:", simple_time)
print("Accuracy:", acc_simple)
print("Loss:", loss_simple)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8932 - loss: 0.3212
SimpleRNN
Training Time: 48.04921817779541
Accuracy: 0.8931860327720642
Loss: 0.3211802542209625


In [21]:
loss_bi, acc_bi = model_bi.evaluate(X_test, y_test)

print("Bidirectional RNN")
print("Training Time:", bi_time)
print("Accuracy:", acc_bi)
print("Loss:", loss_bi)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9724 - loss: 0.0862
Bidirectional RNN
Training Time: 58.47430062294006
Accuracy: 0.9723756909370422
Loss: 0.08624482154846191


In [22]:
from sklearn.metrics import classification_report

y_pred_simple = model.predict(X_test)
y_pred_simple = y_pred_simple.argmax(axis=1)

print("SimpleRNN Report")
print(classification_report(y_test, y_pred_simple))

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
SimpleRNN Report
              precision    recall  f1-score   support

           0       0.89      0.89      0.89       266
           1       0.90      0.90      0.90       277

    accuracy                           0.89       543
   macro avg       0.89      0.89      0.89       543
weighted avg       0.89      0.89      0.89       543



In [23]:
y_pred_bi = model_bi.predict(X_test)
y_pred_bi = y_pred_bi.argmax(axis=1)

print("Bidirectional RNN Report")
print(classification_report(y_test, y_pred_bi))

17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step
Bidirectional RNN Report
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       266
           1       0.97      0.97      0.97       277

    accuracy                           0.97       543
   macro avg       0.97      0.97      0.97       543
weighted avg       0.97      0.97      0.97       543



In [24]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["SimpleRNN", "Bidirectional RNN"],
    "Training Time (sec)": [simple_time, bi_time],
    "Accuracy": [acc_simple, acc_bi],
    "Loss": [loss_simple, loss_bi]
})

print(comparison)

               Model  Training Time (sec)  Accuracy      Loss
0          SimpleRNN            48.049218  0.893186  0.321180
1  Bidirectional RNN            58.474301  0.972376  0.086245


In [25]:
model.save("data/simple_rnn.keras")
model_bi.save("data/bidirectional_rnn.keras")

In [26]:
import pickle

with open("data/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("data/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)